# 📄 Yol Haritası 4. Adım — Gerçek Dökümanlardan Bilgi Tabanı

**Amaç:** Asistanın bilgi tabanını, elle yazılan konular yerine gerçek bir veteriner
dökümanından (PDF) otomatik oluşturmak. Böylece asistan çok daha geniş ve güvenilir
bir kaynaktan cevap verir.

**Adımlar:**
1. PDF dökümanını okuyup metnini çıkarma
2. Metni anlamlı parçalara ayırma (chunking)
3. Parçaları embedding'e çevirip yeni bilgi tabanı kurma
4. Getirmenin (retrieval) doğru çalıştığını test etme

In [1]:
!pip install pypdf sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.9 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer, util
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Embedding modeli hazır.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding modeli hazır.


bu ksıım kütüphaneler ve embedding modeli kısmıydı .


In [3]:
from google.colab import files
yuklenen = files.upload()

Saving 201681633411066886538266.pdf to 201681633411066886538266.pdf


In [4]:
from pypdf import PdfReader

reader = PdfReader(list(yuklenen.keys())[0])
tum_metin = ""
for sayfa in reader.pages:
    m = sayfa.extract_text()
    if m:
        tum_metin += m + "\n"

print("Sayfa:", len(reader.pages), "| Metin:", len(tum_metin), "karakter")

Sayfa: 68 | Metin: 179029 karakter


In [7]:
import re

def parcala(metin, boyut=500, ortusme=50):
    # Önce metni temizle: fazla boşluk ve satır sonlarını düzelt
    metin = re.sub(r'\s+', ' ', metin).strip()

    parcalar = []
    i = 0
    while i < len(metin):
        son = i + boyut
        parca = metin[i:son]
        # Kelime ortasından kesmemek için: son boşluğa kadar geri git
        if son < len(metin):
            bosluk = parca.rfind(" ")
            if bosluk > 100:
                parca = parca[:bosluk]
                son = i + bosluk
        parca = parca.strip()
        if len(parca) > 100:
            parcalar.append(parca)
        i = son - ortusme

    return parcalar

bilgi_tabani = parcala(tum_metin)
bilgi_embed = embed_model.encode(bilgi_tabani, convert_to_tensor=True)
print("Parça sayısı:", len(bilgi_tabani))
print("\nÖrnek parça:\n", bilgi_tabani[50][:400])

Parça sayısı: 395

Örnek parça:
 sıcaklıklarını kısmen tolare edebilir. Ancak gece sıcaklığının yüksek seyir etmesi, yine yüksek sıcaklığa yüksek nemin eşlik etmesi halinde terleme yolu ile ısı kaybı mekanizmasının etkinliğini kaybetmektedir. Sıcak stresi inekte; vücut ısısının artmasının yanında, 10 hayvandan 7 sinin solunum sayısının dakikada 80’ni aşması, yem tüketiminde isteksizlik, yem seçme, salya artışı, süt 7 veriminde az


`re.sub(r'\s+', ' ', metin)` ile metindeki fazla boşluk/satır sonlarını temizliyoruz (PDF'ten gelen dağınıklık gidiyor). İkincisi, `rfind(" ")` ile her parçayı son boşluğa kadar kısaltıyoruz — yani kelime ortasından değil, kelime bitiminden kesiyor. Böylece "ığı için" gibi yarım kelime başlangıçları olmayacak, parçalar düzgün kelimeyle başlayıp bitecek.

In [9]:
soru = "süt sığırlarında hayvan refahı nasıl sağlanır?"
soru_embed = embed_model.encode(soru, convert_to_tensor=True)
benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
print("EN YAKIN PARÇA:\n", bilgi_tabani[benzerlikler.argmax().item()][:400])

EN YAKIN PARÇA:
 onun üçüncü dönemi hayvanın bakım ve beslenmesinin en kolay yü rütülebildiği dönemdir. Bu dönemdeki problem hayvanın besin maddesi ve enerji gereksinimlerinin karşılanamaması değil, hayvanın aşırı beslenmesi ve yağlandırılmasıdır. Bu nedenle ineğin süt verimi çok iyi takip edilmeli ve süt verimi azaldıkça verilen yem miktarı da azaltılarak hayvanın yağlanması önlenmelidir. Süt veriminde aşırı ya d


In [10]:
soru = "ineklerde aşırı yağlanma nasıl önlenir?"
soru_embed = embed_model.encode(soru, convert_to_tensor=True)
benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
print("EN YAKIN PARÇA:\n", bilgi_tabani[benzerlikler.argmax().item()][:400])

EN YAKIN PARÇA:
 edirilmesi: İneklerin aldıkları bu gibi t aze kaba yemlerdeki selüloz oranının düşük olması ve aynı zamanda da süt veriminde görülen artış nedeniyle süt yağında azalma görülür. Bu nedenle meraya çıkan ineklere günde yaklaşık 2 kg kuru ot takviyesi yapılmalıdır. Doymamış yağların ve by-pass yağların yedirilmesi: Özellikle doymamış yağlar işkembedeki sindirimde önemli ölçüde değişikliğe neden olarak


In [14]:
from transformers import pipeline
import torch
# Daha küçük bir model kullanmak için Qwen2-0.5B-Instruct modeline geçiş yapıldı.
# Eğer hala sorun yaşanırsa, "Çalışma Zamanı > Kaynakları Yönet" kısmından GPU kullanımınızı kontrol edin.
llm = pipeline("text-generation", model="Qwen/Qwen2-0.5B-Instruct",
               torch_dtype=torch.float16, device_map="auto")
print("Model yüklendi.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model yüklendi.


Asistanımızın konuşan kısmını-yani dil modelini(LLM) yüklüyor.

`import torch` — PyTorch, modelin arka planda çalışması için gereken kütüphane (matematik/hesaplama motoru gibi).

`llm = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct", ...)`Burada Qwen2.5-3B adlı dil modelini yüklüyoruz. Bu model, senin RAG asistanının "cevap yazan" beyni. "text-generation" = "metin üret" görevi, yani soruya cevap yazma işi.

`torch_dtype=torch.float16 `— modeli daha az bellek kullanacak şekilde yüklüyor (hız/verim için).


In [16]:
def dokuman_asistani(soru):
    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
    en_iyi = benzerlikler.argsort(descending=True)[:2]
    kaynak = "\n\n".join([bilgi_tabani[i] for i in en_iyi])
    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. Sana verilen döküman bilgisine dayanarak "
         "sade ve anlaşılır bir cevap ver. Teşhis koyma, veteriner hekime yönlendir."},
        {"role": "user", "content": f"Döküman bilgisi:\n{kaynak}\n\nSoru: {soru}"}
    ]
    try:
        print("LLM çağrısı başlatılıyor...")
        # max_new_tokens değeri azaltıldı ve max_length açıkça None olarak ayarlandı
        cevap = llm(mesaj, max_new_tokens=100, do_sample=False, max_length=None)
        print("LLM çağrısı tamamlandı. Gelen ham cevap:", type(cevap), cevap)

        if isinstance(cevap, list) and len(cevap) > 0 and 'generated_text' in cevap[0] and isinstance(cevap[0]['generated_text'], list) and len(cevap[0]['generated_text']) > 0:
            for item in reversed(cevap[0]['generated_text']):
                if isinstance(item, dict) and item.get('role') == 'assistant':
                    return item.get('content', 'Asistan cevabı boş.')
        return "Hata: Cevap beklenen formatta değil veya asistan cevabı bulunamadı."
    except Exception as e:
        return f"Hata oluştu: {e}. Lütfen GPU belleğini kontrol edin veya daha küçük bir model kullanmayı deneyin."

print(dokuman_asistani("ineklerde aşırı yağlanma nasıl önlenir?"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM çağrısı başlatılıyor...
LLM çağrısı tamamlandı. Gelen ham cevap: <class 'list'> [{'generated_text': [{'role': 'system', 'content': 'Sen bir veteriner asistanısın. Sana verilen döküman bilgisine dayanarak sade ve anlaşılır bir cevap ver. Teşhis koyma, veteriner hekime yönlendir.'}, {'role': 'user', 'content': 'Döküman bilgisi:\nedirilmesi: İneklerin aldıkları bu gibi t aze kaba yemlerdeki selüloz oranının düşük olması ve aynı zamanda da süt veriminde görülen artış nedeniyle süt yağında azalma görülür. Bu nedenle meraya çıkan ineklere günde yaklaşık 2 kg kuru ot takviyesi yapılmalıdır. Doymamış yağların ve by-pass yağların yedirilmesi: Özellikle doymamış yağlar işkembedeki sindirimde önemli ölçüde değişikliğe neden olarak süt yağını düşürür. 26 Rasyondaki toplam yağ kapsamı % 6 ‘yı geçmemelidir. Buna karşın don yağı,\n\nde yüksek çevre sıcaklığına maruz kalan ineklerde; meme gelişiminin olumsuz etkilenmesinden dolayı, sürekli serinletme sisteminde barındırılan ineklere göre %13,6 dah

dökümandan 2 parça (3 değil, çünkü 2 daha az karıştırıyor) alıp dil modeline veriyoruz, model bunlardan sade bir cevap üretiyor. İşte bu asistanın gerçek cevabı olacak — ham parça değil, modelin yorumu.